## Сбор и валидация бектест данных спреда

Для последующего применения в стратегическую модель 1


In [2]:
import asyncio
import math
import os
from pathlib import Path
from datetime import datetime, timedelta, timezone

import ccxt.async_support as ccxt
import numpy as np
import pandas as pd

UNIVERSE_CSV = "/Users/mishatrubik/Desktop/spread/output/bybit_okx_universe.csv"
OUT_ROOT = Path("/Users/mishatrubik/Desktop/spread/output/spreads_parquet_back")

TIMEFRAME = "1m"
FOLD_COUNT = 5
DAY_STEP = 2
CANDLES_PER_CALL = 1000

START_DAY_UTC = "2026-07-01" 

def build_fold_days(start_day_utc: str, fold_count: int = 5, day_step: int = 2):
    start = pd.Timestamp(start_day_utc, tz="UTC")
    return [start + pd.Timedelta(days=i * day_step) for i in range(fold_count)]

FOLD_DAYS = build_fold_days(START_DAY_UTC, FOLD_COUNT, DAY_STEP)

universe = pd.read_csv(UNIVERSE_CSV)
universe = universe.drop_duplicates(subset=["base_coin"]).reset_index(drop=True)
universe.head()

async def make_exchanges():
    okx = ccxt.okx({
        "enableRateLimit": True,
        "options": {
            "defaultType": "swap",
        },
    })

    bybit = ccxt.bybit({
        "enableRateLimit": True,
        "options": {
            "defaultType": "swap",
        },
    })

    await okx.load_markets()
    await bybit.load_markets()
    return okx, bybit

def day_bounds_ms(day_ts: pd.Timestamp):
    start_ms = int(day_ts.timestamp() * 1000)
    end_ms = int((day_ts + pd.Timedelta(days=1)).timestamp() * 1000)
    return start_ms, end_ms

async def fetch_ohlcv_day(exchange, symbol: str, day_ts: pd.Timestamp, limit: int = 1000):
    start_ms, end_ms = day_bounds_ms(day_ts)
    tf_ms = 60_000

    rows = []
    since = start_ms

    while since < end_ms:
        params = {}
        if exchange.id == "bybit":
            params["category"] = "linear"

        batch = await exchange.fetch_ohlcv(
            symbol,
            timeframe=TIMEFRAME,
            since=since,
            limit=limit,
            params=params,
        )

        if not batch:
            break

        rows.extend(batch)

        last_ts = batch[-1][0]
        next_since = last_ts + tf_ms

        if next_since <= since:
            break

        since = next_since

        if len(batch) < limit:
            break

    if not rows:
        return pd.DataFrame(columns=["ts", "open", "high", "low", "close", "volume"])

    df = pd.DataFrame(rows, columns=["ts", "open", "high", "low", "close", "volume"])
    df = df.drop_duplicates(subset=["ts"]).sort_values("ts").reset_index(drop=True)
    df = df[(df["ts"] >= start_ms) & (df["ts"] < end_ms)].copy()
    df["dt"] = pd.to_datetime(df["ts"], unit="ms", utc=True)
    return df

async def preflight_symbol_day(exchange, symbol: str, day_ts: pd.Timestamp):
    start_ms, _ = day_bounds_ms(day_ts)
    params = {}
    if exchange.id == "bybit":
        params["category"] = "linear"

    try:
        batch = await exchange.fetch_ohlcv(
            symbol,
            timeframe=TIMEFRAME,
            since=start_ms,
            limit=5,
            params=params,
        )
        if not batch:
            return False, "empty_at_day_start"
        return True, "ok"
    except Exception as e:
        msg = str(e).lower()
        if "symbol" in msg or "not found" in msg or "does not exist" in msg:
            return False, "symbol_not_available"
        if "before" in msg or "listing" in msg or "invalid" in msg:
            return False, "before_listing_or_invalid_range"
        return False, f"error:{type(e).__name__}"

def validate_and_merge(okx_df: pd.DataFrame, bybit_df: pd.DataFrame, base_coin: str, day_ts: pd.Timestamp):
    expected_index = pd.date_range(day_ts, day_ts + pd.Timedelta(days=1) - pd.Timedelta(minutes=1), freq="1min", tz="UTC")

    okx = okx_df.set_index("dt")[["open", "high", "low", "close", "volume"]].reindex(expected_index)
    bybit = bybit_df.set_index("dt")[["open", "high", "low", "close", "volume"]].reindex(expected_index)

    okx.columns = [f"okx_{c}" for c in okx.columns]
    bybit.columns = [f"bybit_{c}" for c in bybit.columns]

    merged = pd.concat([okx, bybit], axis=1).reset_index().rename(columns={"index": "event_dt"})
    merged["base_coin"] = base_coin

    merged["okx_present"] = merged["okx_close"].notna().astype(int)
    merged["bybit_present"] = merged["bybit_close"].notna().astype(int)
    merged["both_present"] = ((merged["okx_present"] == 1) & (merged["bybit_present"] == 1)).astype(int)

    merged["spread_close_bps"] = np.where(
        merged["both_present"] == 1,
        (merged["bybit_close"] / merged["okx_close"] - 1.0) * 1e4,
        np.nan,
    )

    merged["spread_long_proxy_bps"] = np.where(
        merged["okx_low"].notna() & merged["bybit_high"].notna(),
        (merged["bybit_high"] / merged["okx_low"] - 1.0) * 1e4,
        np.nan,
    )

    merged["spread_short_proxy_bps"] = np.where(
        merged["bybit_low"].notna() & merged["okx_high"].notna(),
        (merged["okx_high"] / merged["bybit_low"] - 1.0) * 1e4,
        np.nan,
    )

    stats = {
        "base_coin": base_coin,
        "date": str(day_ts.date()),
        "n_expected": len(expected_index),
        "n_okx_raw": len(okx_df),
        "n_bybit_raw": len(bybit_df),
        "n_both_present": int(merged["both_present"].sum()),
        "okx_coverage_ratio": float(merged["okx_present"].mean()),
        "bybit_coverage_ratio": float(merged["bybit_present"].mean()),
        "both_coverage_ratio": float(merged["both_present"].mean()),
    }

    return merged, stats
    
async def collect_symbol_day(okx, bybit, row, day_ts):
    base_coin = str(row["base_coin"]).strip()
    symbol = row.get("ccxt_symbol", f"{base_coin}/USDT:USDT")

    okx_ok, okx_reason = await preflight_symbol_day(okx, symbol, day_ts)
    if not okx_ok:
        return None, {
            "base_coin": base_coin,
            "date": str(day_ts.date()),
            "status": "skip",
            "reason": f"okx:{okx_reason}",
        }

    bybit_ok, bybit_reason = await preflight_symbol_day(bybit, symbol, day_ts)
    if not bybit_ok:
        return None, {
            "base_coin": base_coin,
            "date": str(day_ts.date()),
            "status": "skip",
            "reason": f"bybit:{bybit_reason}",
        }

    try:
        okx_df, bybit_df = await asyncio.gather(
            fetch_ohlcv_day(okx, symbol, day_ts, limit=CANDLES_PER_CALL),
            fetch_ohlcv_day(bybit, symbol, day_ts, limit=CANDLES_PER_CALL),
        )
    except Exception as e:
        return None, {
            "base_coin": base_coin,
            "date": str(day_ts.date()),
            "status": "error",
            "reason": f"fetch:{type(e).__name__}:{str(e)[:200]}",
        }

    if okx_df.empty or bybit_df.empty:
        return None, {
            "base_coin": base_coin,
            "date": str(day_ts.date()),
            "status": "skip",
            "reason": f"empty_after_fetch okx={len(okx_df)} bybit={len(bybit_df)}",
        }

    merged, stats = validate_and_merge(okx_df, bybit_df, base_coin, day_ts)
    stats["status"] = "ok"
    stats["reason"] = "ok"
    return merged, stats

async def collect_fold(day_ts: pd.Timestamp, fold_idx: int, universe_df: pd.DataFrame):
    fold_dir = OUT_ROOT / f"fold_{fold_idx:02d}" / f"date={day_ts.date()}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    okx, bybit = await make_exchanges()
    manifest_rows = []

    try:
        for _, row in universe_df.iterrows():
            merged, stats = await collect_symbol_day(okx, bybit, row, day_ts)
            manifest_rows.append(stats)

            if merged is not None and not merged.empty:
                out_file = fold_dir / f"{row['base_coin']}.parquet"
                merged.to_parquet(out_file, index=False)

    finally:
        await okx.close()
        await bybit.close()

    manifest = pd.DataFrame(manifest_rows)
    manifest.to_parquet(fold_dir / "_manifest.parquet", index=False)
    manifest.to_csv(fold_dir / "_manifest.csv", index=False)

    return manifest

all_manifests = []

for fold_idx, day_ts in enumerate(FOLD_DAYS, start=1):
    print(f"start fold={fold_idx} day={day_ts.date()}")
    manifest = await collect_fold(day_ts, fold_idx, universe)
    all_manifests.append(manifest)

summary = pd.concat(all_manifests, ignore_index=True)
summary.to_parquet(OUT_ROOT / "_summary.parquet", index=False)
summary.to_csv(OUT_ROOT / "_summary.csv", index=False)
summary.head()

start fold=1 day=2026-07-01


CancelledError: 

In [4]:
from pathlib import Path
import pandas as pd

BACK_ROOT = Path("/Users/mishatrubik/Desktop/spread/output/spreads_parquet_back")

def load_back_parquet(base_coin: str, fold_idx: int, date_str: str) -> pd.DataFrame:
    """
    base_coin: 'ADA', 'LRC', ...
    fold_idx: 1..5
    date_str: '2026-07-01' (тот день, который ты использовал)
    """
    fold_dir = BACK_ROOT / f"fold_{fold_idx:02d}" / f"date={date_str}"
    file_path = fold_dir / f"{base_coin}.parquet"
    print("reading:", file_path)
    df = pd.read_parquet(file_path)
    df["event_dt"] = pd.to_datetime(df["event_dt"], utc=True)
    df = df.sort_values("event_dt").reset_index(drop=True)
    return df
def inspect_back_df(df: pd.DataFrame):
    print("shape:", df.shape)
    print("columns:", df.columns.tolist())
    print("time min:", df["event_dt"].min(), "time max:", df["event_dt"].max())
    print("okx_coverage:", df["okx_present"].mean())
    print("bybit_coverage:", df["bybit_present"].mean())
    print("both_coverage:", df["both_present"].mean())
    print(df[["event_dt", "okx_close", "bybit_close", "spread_close_bps"]].head())
import plotly.graph_objects as go

def plot_back_spread(df: pd.DataFrame, base_coin: str, title_suffix: str = ""):
    df = df.sort_values("event_dt").copy()
    
    fig = go.Figure()

    # спред по close
    fig.add_trace(
        go.Scatter(
            x=df["event_dt"],
            y=df["spread_close_bps"],
            mode="lines+markers",
            name="spread_close_bps",
            line=dict(width=1.4, color="#1f77b4"),
            marker=dict(size=2.5),
        )
    )

    # опционально — отдельные прокси
    if "spread_long_proxy_bps" in df.columns:
        fig.add_trace(
            go.Scatter(
                x=df["event_dt"],
                y=df["spread_long_proxy_bps"],
                mode="lines",
                name="spread_long_proxy_bps",
                line=dict(width=1.1, color="#d62728"),
            )
        )

    if "spread_short_proxy_bps" in df.columns:
        fig.add_trace(
            go.Scatter(
                x=df["event_dt"],
                y=df["spread_short_proxy_bps"],
                mode="lines",
                name="spread_short_proxy_bps",
                line=dict(width=1.1, color="#2ca02c"),
            )
        )

    fig.update_layout(
        title=f"{base_coin} backtest spreads {title_suffix}",
        width=1400,
        height=700,
        xaxis_title="event_dt (UTC)",
        yaxis_title="spread bps",
        hovermode="x unified",
        legend=dict(font=dict(size=11)),
        margin=dict(l=50, r=20, t=60, b=40),
    )
    fig.show()
def plot_back_prices(df: pd.DataFrame, base_coin: str, title_suffix: str = ""):
    df = df.sort_values("event_dt").copy()

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=df["event_dt"],
            y=df["okx_close"],
            mode="lines",
            name="okx_close",
            line=dict(width=1.2, color="#1f77b4"),
        )
    )

    fig.add_trace(
        go.Scatter(
            x=df["event_dt"],
            y=df["bybit_close"],
            mode="lines",
            name="bybit_close",
            line=dict(width=1.2, color="#ff7f0e"),
        )
    )

    fig.update_layout(
        title=f"{base_coin} okx vs bybit close {title_suffix}",
        width=1400,
        height=700,
        xaxis_title="event_dt (UTC)",
        yaxis_title="price",
        hovermode="x unified",
        legend=dict(font=dict(size=11)),
        margin=dict(l=50, r=20, t=60, b=40),
    )
    fig.show()

base_coin = "0G"
fold_idx = 1
date_str = "2026-07-01"  # день этого fold

df_back = load_back_parquet(base_coin, fold_idx, date_str)
inspect_back_df(df_back)

plot_back_prices(df_back, base_coin, title_suffix=f"fold {fold_idx} {date_str}")
plot_back_spread(df_back, base_coin, title_suffix=f"fold {fold_idx} {date_str}")

reading: /Users/mishatrubik/Desktop/spread/output/spreads_parquet_back/fold_01/date=2026-07-01/0G.parquet
shape: (1440, 18)
columns: ['event_dt', 'okx_open', 'okx_high', 'okx_low', 'okx_close', 'okx_volume', 'bybit_open', 'bybit_high', 'bybit_low', 'bybit_close', 'bybit_volume', 'base_coin', 'okx_present', 'bybit_present', 'both_present', 'spread_close_bps', 'spread_long_proxy_bps', 'spread_short_proxy_bps']
time min: 2026-07-01 00:00:00+00:00 time max: 2026-07-01 23:59:00+00:00
okx_coverage: 0.06944444444444445
bybit_coverage: 1.0
both_coverage: 0.06944444444444445
                   event_dt  okx_close  bybit_close  spread_close_bps
0 2026-07-01 00:00:00+00:00     0.2024       0.2026          9.881423
1 2026-07-01 00:01:00+00:00     0.2022       0.2025         14.836795
2 2026-07-01 00:02:00+00:00     0.2023       0.2024          4.943154
3 2026-07-01 00:03:00+00:00     0.2023       0.2025          9.886307
4 2026-07-01 00:04:00+00:00     0.2023       0.2028         24.715769
